In [1]:
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
import os
import numpy as np
pd.set_option('display.max_columns', None)

In [12]:
df_errores= pd.read_excel('c:/data/tabla_errores2.xlsx', sheet_name='Sheet1', dtype={'CODIGO_ERROR': str, 'IDEERROR': str})
df_errores.loc[df_errores['CODIGO_ERROR'].isna(), 'CODIGO_ERROR'] = ""
df_errores.head(3)

,IDEERROR,CODIGO_ERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,0974,1068,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO...,CONFIGURACION SAS,ANULAR MASIVAS
1,0005,0003,ERROR EN CARGA DE TRAMA,ERROR EN CARGA DE TRAMA,ANALISIS EMISOR,NaN
2,1158,1155,TRAMA REPETIDA O DUPLICADA,TRAMA REPETIDA O DUPLICADA,CONFIGURACION SAS,ANULAR MASIVAS


In [2]:
df_desgravamen= pd.read_excel('C:/data/408 - desgravamen 01.xlsx', sheet_name='Sheet1', 
                              dtype={'COD DE CERTIFICADO': str, 'DESCRIPCION ERROR': str, 
                                     'IDELOTE': str, 'CODIGO ERROR': str, 
                                     'SUMA ASEGURADA':str, 'TASA': str, 
                                     'TASA RECARGO': str, 'PRIMABRUTACAN':str, 'PRIMANETACAN': str})

In [3]:
df_desgravamen.columns = (df_desgravamen.columns.str.strip()
                          .str.upper()  # opcional: todo en mayúsculas
                          .str.replace(r'[^A-Za-z0-9_]', '_', regex=True)  # reemplazar estapacios por _
                        )

df_desgravamen.drop(['IDELOTE','IDEDET_1','DESCRIPCION_ERROR'], axis=1, inplace=True)
df_desgravamen= df_desgravamen.rename(columns={'CODIGO_ERROR': 'IDEERROR'})

In [4]:
df_desgravamen= df_desgravamen.fillna({'LOTES_ANTERIORES':'', 'MONEDA': 'SIN DATO', 'ORIGEN_ERROR': 'SIN DATO', 'TIPDOCUMENTO':'SIN DATO'})
df_desgravamen.loc[df_desgravamen['MONEDA'] == 'nan', 'MONEDA'] = 'SIN DATO'
df_desgravamen.loc[df_desgravamen['ORIGEN_ERROR'] == 'nan', 'ORIGEN_ERROR'] = 'SIN DATO'

In [5]:
df_desgravamen['SUMA_ASEGURADA'] = df_desgravamen['SUMA_ASEGURADA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA'] = df_desgravamen['TASA'].str.replace(',', '.', regex=False)
df_desgravamen['TASA_RECARGO'] = df_desgravamen['TASA_RECARGO'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMABRUTACAN'] = df_desgravamen['PRIMABRUTACAN'].str.replace(',', '.', regex=False)
df_desgravamen['PRIMANETACAN'] = df_desgravamen['PRIMANETACAN'].str.replace(',', '.', regex=False)

In [6]:
def limpiar_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

In [8]:
df_desgravamen["LOTES_ANTERIORES"] = df_desgravamen["LOTES_ANTERIORES"].astype(str)
df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], format="%d/%m/%Y %I:%M:%S %p", errors='coerce')
df_desgravamen['CODIGO_PRODUCTO'] = df_desgravamen['CODIGO_PRODUCTO'].fillna(0).astype('int')
df_desgravamen["PRODUCTO"] = df_desgravamen["PRODUCTO"].astype(str)
df_desgravamen['CODIGO_PLAN'] = df_desgravamen['CODIGO_PLAN'].fillna(0).astype('int')
df_desgravamen["NOMBRE_DE_PLAN"] = df_desgravamen["NOMBRE_DE_PLAN"].astype(str)
df_desgravamen["COD_DE_CERTIFICADO"] = df_desgravamen["COD_DE_CERTIFICADO"].astype(str)
df_desgravamen['FECINI_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECINI_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FECFIN_ALTA_CERTIFICADO'] = pd.to_datetime(df_desgravamen['FECFIN_ALTA_CERTIFICADO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen["TIPO_MOVIMIENTO"] = df_desgravamen["TIPO_MOVIMIENTO"].astype(str)
df_desgravamen['FEC__INICIO'] = pd.to_datetime(df_desgravamen['FEC__INICIO'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['FEC__FIN'] = pd.to_datetime(df_desgravamen['FEC__FIN'], format='%d/%m/%Y', errors='coerce')
df_desgravamen['MONEDA'] = df_desgravamen['MONEDA'].astype(str)
df_desgravamen['SUMA_ASEGURADA'] = pd.to_numeric(df_desgravamen['SUMA_ASEGURADA'], errors="coerce").astype('float64')
df_desgravamen['TASA'] = pd.to_numeric(df_desgravamen['TASA'], errors="coerce").astype('float64')
df_desgravamen['TASA_RECARGO'] = pd.to_numeric(df_desgravamen['TASA_RECARGO'], errors="coerce").astype('float64')
df_desgravamen['PRIMABRUTACAN'] = pd.to_numeric(df_desgravamen['PRIMABRUTACAN'], errors="coerce").astype('float64')
df_desgravamen['PRIMANETACAN'] = pd.to_numeric(df_desgravamen['PRIMANETACAN'], errors="coerce").astype('float64')
df_desgravamen['NOMCOMPLETO'] = df_desgravamen['NOMCOMPLETO'].astype(str)
df_desgravamen['APEPATERNO'] = df_desgravamen['APEPATERNO'].astype(str)
df_desgravamen['APEMATERNO'] = df_desgravamen['APEMATERNO'].astype(str)
df_desgravamen["FECNACIMIENTO"] = limpiar_fecha(df_desgravamen["FECNACIMIENTO"])
df_desgravamen["TIPDOCUMENTO"] = df_desgravamen["TIPDOCUMENTO"].astype(str)
df_desgravamen['NUMDOCUMENTO'] = df_desgravamen['NUMDOCUMENTO'].astype(str)
df_desgravamen['NOMBRE_DE_ARCHIVO'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].astype(str)
df_desgravamen['LINEA_TRAMA'] = df_desgravamen['LINEA_TRAMA'].astype(str)
df_desgravamen['ORIGEN_ERROR'] = df_desgravamen['ORIGEN_ERROR'].astype(str)
df_desgravamen['IDEERROR'] = df_desgravamen['IDEERROR'].apply(lambda x: str(int(x)).zfill(4) if pd.notnull(x) else "")

In [7]:
#### FUNCION PARA EXTRAER LA FECHA DEL NOMBRE DE ARCHIVO
def extraer_fecha(fecha):
    try:
        fecha_str = fecha.split("_")[2]
        return datetime.strptime(fecha_str, "%Y%m%d").date()
    except Exception:
        return None  # En caso de error

#### FUNCION PARA CLASIFICAR EL REGISTRO
def clasificar(row):
    # Si lote no es nulo
    if str(row['LOTES_ANTERIORES']).strip() != "":
        return 'Regularizacion no Exitosa'

    # Si último número del nombre de archivo > 100
    try:
        ultimo_numero = int(row['NOMBRE_DE_ARCHIVO'].split("_")[-1][:-4])
        if ultimo_numero > 100:
            return 'Regularizacion no Exitosa'
    except:
        pass

    # Si ninguna condición se cumple
    return 'Mes corriente'

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id, schema):
    #bigquery_client = bigquery.Client()
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    try:
        tabla = bigquery_client.get_table(table_ref)
        tabla_existe = True
    except:
        tabla_existe = False

    if not tabla_existe:
        # Crear la tabla si no existe
        tabla = bigquery.Table(table_ref, schema=schema)
        tabla = bigquery_client.create_table(tabla)
        print(f'----- Se ha creado la tabla {table_id} en el dataset {dataset_id} -----')

    # Agregar los registros de data a la tabla existente o recién creada
    job_config = bigquery.LoadJobConfig()
    job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND if tabla_existe else bigquery.WriteDisposition.WRITE_TRUNCATE
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    print(f'----- REGISTROS AGREGADOS CORRECTAMENTE EN: {table_id} -------')
    return

In [13]:
df_desgravamen= df_desgravamen[df_desgravamen['PRIMABRUTACAN']<20000]   #Eliminar las filas con valores atipicos muy altos
df_desgravamen['FECHA_TRAMA'] = df_desgravamen['NOMBRE_DE_ARCHIVO'].apply(extraer_fecha)    # Crear nuevas columnas cextrayendo la fecha del nombre de archivo
df_desgravamen['FECHA_TRAMA']= pd.to_datetime(df_desgravamen['FECHA_TRAMA'], errors='coerce')
#df_desgravamen['FECHA_PERIODO']= pd.to_datetime(FECHA_PERIODO)
df_desgravamen['TIPO_ERROR']= df_desgravamen.apply(clasificar, axis=1)    # Crear colunma evaluando condiciones en el contenido de otras columnas

In [15]:
df_desgravamen= df_desgravamen.merge(df_errores, on='IDEERROR', how='left')
df_desgravamen.drop(['IDEERROR'], axis=1, inplace=True)
#df_desgravamen= df_desgravamen.merge(df_errores, on='CODIGO_ERROR', how='left')
df_desgravamen.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,FECHA_TRAMA,TIPO_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,43348,,2015-05-26,3665,Desgravamen BBVA KT,75889,Desgravamen de Capital de Trabajo BBVA Contine...,00110242524000046142,NaT,NaT,77771303,Exclusion,NaT,2014-11-12,SOL,0.0,0.0,0.0,0.0,0.0,NORA ELIZABETH,ALAVE,CUTIPA,1983-05-16,1.0,10417390940,20100130204_0155001_20141101_003.TXT,919001102425240000461420011024254500008398201P...,Error canal,2014-11-01,Mes corriente,NaN,NaN,NaN,NaN,NaN
1,43348,,2015-05-26,3665,Desgravamen BBVA KT,75889,Desgravamen de Capital de Trabajo BBVA Contine...,00110316714000135600,NaT,NaT,77771306,Exclusion,NaT,2014-11-26,SOL,0.0,0.0,0.0,0.0,0.0,LUIS HUMBERTO,TORRES,ROJAS,1968-03-14,1.0,10165865150,20100130204_0155001_20141101_003.TXT,919001103167140001356000011031670500042760801P...,Error canal,2014-11-01,Mes corriente,NaN,NaN,NaN,NaN,NaN
2,43348,,2015-05-26,3665,Desgravamen BBVA KT,75889,Desgravamen de Capital de Trabajo BBVA Contine...,00110174024000154221,NaT,NaT,77771301,Exclusion,NaT,2014-11-26,SOL,0.0,0.0,0.0,0.0,0.0,nan,CORPORACION EMILY SRL,nan,2010-07-23,1.0,20536912852,20100130204_0155001_20141101_003.TXT,919001101740240001542210011017400500081766901P...,Error canal,2014-11-01,Mes corriente,NaN,NaN,NaN,NaN,NaN


In [16]:
df_desgravamen.head(3)

,NRO_LOTE,LOTES_ANTERIORES,FECHA_CARGA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_PLAN,NOMBRE_DE_PLAN,COD_DE_CERTIFICADO,FECINI_ALTA_CERTIFICADO,FECFIN_ALTA_CERTIFICADO,IDEDET,TIPO_MOVIMIENTO,FEC__INICIO,FEC__FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_RECARGO,PRIMABRUTACAN,PRIMANETACAN,NOMCOMPLETO,APEPATERNO,APEMATERNO,FECNACIMIENTO,TIPDOCUMENTO,NUMDOCUMENTO,NOMBRE_DE_ARCHIVO,LINEA_TRAMA,ORIGEN_ERROR,FECHA_TRAMA,TIPO_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR,DETALLE,SOLUCION,SOLUCION_2
0,43348,,2015-05-26,3665,Desgravamen BBVA KT,75889,Desgravamen de Capital de Trabajo BBVA Contine...,00110242524000046142,NaT,NaT,77771303,Exclusion,NaT,2014-11-12,SOL,0.0,0.0,0.0,0.0,0.0,NORA ELIZABETH,ALAVE,CUTIPA,1983-05-16,1.0,10417390940,20100130204_0155001_20141101_003.TXT,919001102425240000461420011024254500008398201P...,Error canal,2014-11-01,Mes corriente,NaN,NaN,NaN,NaN,NaN
1,43348,,2015-05-26,3665,Desgravamen BBVA KT,75889,Desgravamen de Capital de Trabajo BBVA Contine...,00110316714000135600,NaT,NaT,77771306,Exclusion,NaT,2014-11-26,SOL,0.0,0.0,0.0,0.0,0.0,LUIS HUMBERTO,TORRES,ROJAS,1968-03-14,1.0,10165865150,20100130204_0155001_20141101_003.TXT,919001103167140001356000011031670500042760801P...,Error canal,2014-11-01,Mes corriente,NaN,NaN,NaN,NaN,NaN
2,43348,,2015-05-26,3665,Desgravamen BBVA KT,75889,Desgravamen de Capital de Trabajo BBVA Contine...,00110174024000154221,NaT,NaT,77771301,Exclusion,NaT,2014-11-26,SOL,0.0,0.0,0.0,0.0,0.0,nan,CORPORACION EMILY SRL,nan,2010-07-23,1.0,20536912852,20100130204_0155001_20141101_003.TXT,919001101740240001542210011017400500081766901P...,Error canal,2014-11-01,Mes corriente,NaN,NaN,NaN,NaN,NaN


In [18]:
df_desgravamen['DESCRIPCION_ERROR'].value_counts()

DESCRIPCION_ERROR
ERROR EN ALTA                                                 11911
PAGO PENDIENTE DE CORREGIR                                     6414
CAMPOS EN BLANCO                                               3545
NI ÉXITO NI ERROR                                               984
TRAMA REPETIDA O DUPLICADA                                      818
SIN POLIZA                                                      255
VALIDACIONES CONSECUENCIA ACSEL E                               180
VERIFICACION DE DATOS                                           102
YA EXISTE UN PAGO CON LA MISMA FECHA DE INICIO DE VIGENCIA        1
ERROR EN CARGA DE TRAMA                                           1
Name: count, dtype: int64